In [ ]:

from pathlib import Path
import pandas as pd

# Current working directory
current_dir = Path.cwd()

# Move up to the main project folder
project_path = current_dir.parent

# Find the Olist dataset using the standard folder structure
data_path = project_path / "data" / "olist"

# Check the paths
print("Current directory:", current_dir)
print("Project folder:", project_path)
print("Data folder:", data_path)
print("Data folder exists:", data_path.exists())





In [ ]:

print("Folders inside the project:")

for item in project_path.iterdir():
    print(repr(item.name), "| Folder:", item.is_dir())


In [ ]:

from pathlib import Path
import pandas as pd

# Locate the main project folder
project_path = Path.cwd().parent

# Use the actual Olist folder name
data_path = project_path / "data : olist"

# Locate the orders CSV
orders_path = data_path / "olist_orders_dataset.csv"

# Check whether Python can find the file
print("File exists:", orders_path.exists())

# Load the dataset
orders_df = pd.read_csv(orders_path)

# Display the first five orders
display(orders_df.head())

# Display rows and columns
print("Dataset shape:", orders_df.shape)


In [ ]:

orders_df.info()



In [ ]:

orders_df.isnull().sum()



In [ ]:

orders_df["order_id"].duplicated().sum()


In [ ]:

orders_df["order_status"].value_counts()


In [ ]:

missing_delivery = orders_df[
    orders_df["order_delivered_customer_date"].isna()
]

print("Orders missing delivery date:", len(missing_delivery))

display(
    missing_delivery["order_status"].value_counts()
)


In [ ]:

inconsistent_orders = orders_df[
    (orders_df["order_status"] == "delivered")
    &
    (orders_df["order_delivered_customer_date"].isna())
]

display(inconsistent_orders)

print("Inconsistent records:", len(inconsistent_orders))


In [ ]:

# Select orders with missing delivery dates
missing_delivery = orders_df[
    orders_df["order_delivered_customer_date"].isna()
]

# Count how many orders are affected
print("Missing delivery dates:", len(missing_delivery))

# Check the status of these orders
display(
    missing_delivery["order_status"].value_counts()
)


In [ ]:

inconsistent_orders = orders_df[
    (orders_df["order_status"] == "delivered")
    &
    (orders_df["order_delivered_customer_date"].isna())
]

display(inconsistent_orders)

print("Inconsistent orders:", len(inconsistent_orders))


In [ ]:

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders_df[column] = pd.to_datetime(
        orders_df[column],
        errors="coerce"
    )

print(orders_df[date_columns].dtypes)


In [ ]:

print(orders_df[date_columns].isna().sum())


In [ ]:

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Define PostgreSQL connection details
db_url = URL.create(
    drivername="postgresql+psycopg",
    username="nikhilkoushik",
    host="localhost",
    port=5432,
    database="supply_chain_forensics"
)

# Create the database engine
engine = create_engine(db_url)

# Test the connection
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database(), current_user")
    )

    print(result.fetchone())


In [ ]:

# Check total number of orders
print("Total orders:", len(orders_df))

# Check missing order IDs
print("Missing IDs:", orders_df["order_id"].isna().sum())

# Check duplicate order IDs
print("Duplicate IDs:", orders_df["order_id"].duplicated().sum())

# Check date column types
print(orders_df[date_columns].dtypes)


In [ ]:

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM olist.orders")
    )

    database_count = result.scalar_one()

print("Orders currently in PostgreSQL:", database_count)


In [ ]:

# Prevent duplicate loading
if database_count != 0:
    raise ValueError(
        "The orders table is not empty. Load stopped."
    )

# Verify source data before loading
if (
    len(orders_df) != 99441
    or orders_df["order_id"].isna().any()
    or orders_df["order_id"].duplicated().any()
):
    raise ValueError("Source data validation failed.")

# Insert orders into the existing PostgreSQL table
with engine.begin() as connection:
    orders_df.to_sql(
        name="orders",
        con=connection,
        schema="olist",
        if_exists="append",
        index=False,
        chunksize=1000
    )

print("Data loading completed!")


In [ ]:

with engine.connect() as connection:
    loaded_count = connection.execute(
        text("SELECT COUNT(*) FROM olist.orders")
    ).scalar_one()

print("Python records:", len(orders_df))
print("PostgreSQL records:", loaded_count)
print("Counts match:", len(orders_df) == loaded_count)


In [ ]:

# Locate the customer dataset
customers_path = data_path / "olist_customers_dataset.csv"

# Load the CSV file
customers_df = pd.read_csv(customers_path)

# Display first five records
display(customers_df.head())

# Check dataset dimensions
print("Dataset shape:", customers_df.shape)

# Inspect column names, types and missing values
customers_df.info()


In [ ]:

print(
    "Duplicate customer IDs:",
    customers_df["customer_id"].duplicated().sum()
)

print(
    "Duplicate unique customer IDs:",
    customers_df["customer_unique_id"].duplicated().sum()
)


In [ ]:

print(customers_df.isna().sum())


In [ ]:

print(
    customers_df["customer_state"].value_counts().head(10)
)


In [ ]:

customers_df["zip_prefix_text"] = (
    customers_df["customer_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

display(
    customers_df[
        ["customer_zip_code_prefix", "zip_prefix_text"]
    ].head(10)
)


In [ ]:

print(
    "Duplicate customer IDs:",
    customers_df["customer_id"].duplicated().sum()
)

print(
    "Duplicate unique customer IDs:",
    customers_df["customer_unique_id"].duplicated().sum()
)


In [ ]:

# Create a separate DataFrame for database loading
customers_load_df = customers_df[
    [
        "customer_id",
        "customer_unique_id",
        "zip_prefix_text",
        "customer_city",
        "customer_state"
    ]
].copy()

# Rename standardized ZIP column to match PostgreSQL
customers_load_df = customers_load_df.rename(
    columns={
        "zip_prefix_text": "customer_zip_code_prefix"
    }
)

# Check the resulting structure
display(customers_load_df.head())
print(customers_load_df.shape)


In [ ]:

from sqlalchemy import text

# Check whether customers table already contains records
with engine.connect() as connection:
    current_count = connection.execute(
        text("SELECT COUNT(*) FROM olist.customers")
    ).scalar_one()

print("Existing customer records:", current_count)

# Validate source data and avoid duplicate loading
if current_count != 0:
    raise ValueError("Customers table is not empty.")

if customers_load_df["customer_id"].isna().any():
    raise ValueError("Missing customer IDs found.")

if customers_load_df["customer_id"].duplicated().any():
    raise ValueError("Duplicate customer IDs found.")

# Load all customer records in one transaction
with engine.begin() as connection:
    customers_load_df.to_sql(
        name="customers",
        con=connection,
        schema="olist",
        if_exists="append",
        index=False,
        chunksize=1000
    )

print("Customer loading complete!")


In [ ]:

# Step 1: Locate the order-items CSV
order_items_path = data_path / "olist_order_items_dataset.csv"

# Step 2: Load the data
order_items_df = pd.read_csv(order_items_path)

# Step 3: Display the first five rows
display(order_items_df.head())

# Step 4: Check the shape
print("Dataset shape:", order_items_df.shape)

# Step 5: Inspect the columns and data types
order_items_df.info()


In [ ]:

# Missing values in each column
print("Missing values:")
print(order_items_df.isna().sum())

# Count rows
print("\nTotal order-item records:")
print(len(order_items_df))

# Count unique orders
print("\nUnique orders:")
print(order_items_df["order_id"].nunique())

# Check duplicate combinations of order and item ID
print("\nDuplicate order-item keys:")
print(
    order_items_df.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)


In [ ]:

items_per_order = (
    order_items_df
    .groupby("order_id")
    .size()
)

print("Average items per order:")
print(round(items_per_order.mean(), 2))

print("\nMaximum items in one order:")
print(items_per_order.max())

print("\nOrders by item count:")
print(items_per_order.value_counts().sort_index().head(10))


In [ ]:

# Convert shipping deadline from text to datetime
order_items_df["shipping_limit_date"] = pd.to_datetime(
    order_items_df["shipping_limit_date"],
    errors="coerce"
)

# Verify the converted data type
print(order_items_df["shipping_limit_date"].dtype)

# Check for invalid or missing dates
print(
    "Missing shipping deadlines:",
    order_items_df["shipping_limit_date"].isna().sum()
)


In [ ]:

order_items_df["shipping_limit_date"] = pd.to_datetime(
    order_items_df["shipping_limit_date"],
    errors="coerce"
)

print(order_items_df["shipping_limit_date"].dtype)
print(
    "Missing shipping dates:",
    order_items_df["shipping_limit_date"].isna().sum()
)


In [ ]:

print("Total items:", len(order_items_df))

print(
    "Duplicate item keys:",
    order_items_df.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print(
    "Missing order IDs:",
    order_items_df["order_id"].isna().sum()
)

with engine.connect() as connection:
    existing_count = connection.execute(
        text("SELECT COUNT(*) FROM olist.order_items")
    ).scalar_one()

print("Existing PostgreSQL items:", existing_count)


In [ ]:

# Stop if the table already contains data
if existing_count != 0:
    raise ValueError("Order-items table is not empty.")

# Stop if the composite primary key is duplicated
if order_items_df.duplicated(
    subset=["order_id", "order_item_id"]
).any():
    raise ValueError("Duplicate order-item keys found.")

# Stop if shipping dates are invalid
if order_items_df["shipping_limit_date"].isna().any():
    raise ValueError("Invalid shipping dates found.")

# Insert the records into PostgreSQL
with engine.begin() as connection:
    order_items_df.to_sql(
        name="order_items",
        con=connection,
        schema="olist",
        if_exists="append",
        index=False,
        chunksize=1000
    )

print("Order items loaded successfully!")


In [ ]:

# Locate the products CSV
products_path = data_path / "olist_products_dataset.csv"

# Load the products into Pandas
products_df = pd.read_csv(products_path)

# Preview the first five products
display(products_df.head())

# Check the number of rows and columns
print("Dataset shape:", products_df.shape)

# Inspect columns, types and missing values
products_df.info()


In [ ]:

print("Total product records:", len(products_df))

print(
    "Duplicate product IDs:",
    products_df["product_id"].duplicated().sum()
)

print("\nMissing values:")
print(products_df.isna().sum())

print(
    "\nUnique product categories:",
    products_df["product_category_name"].nunique()
)


In [ ]:

# Identify products without a category
missing_categories = products_df[
    products_df["product_category_name"].isna()
]

print("Products without categories:", len(missing_categories))

# Identify products missing any physical dimension or weight
physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

missing_physical = products_df[
    products_df[physical_columns].isna().any(axis=1)
]

print("Products missing physical attributes:", len(missing_physical))

# Check whether the 610 missing category names
# correspond to the same missing descriptive attributes
description_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

print(
    "Products missing all four descriptive attributes:",
    products_df[description_columns].isna().all(axis=1).sum()
)


In [ ]:

for column in physical_columns:
    print(
        column,
        "Zero or negative values:",
        (products_df[column] <= 0).sum()
    )

display(products_df[physical_columns].describe())


In [ ]:

# Create a separate copy for PostgreSQL loading
products_load_df = products_df.copy()

# Identify columns that should contain whole numbers
integer_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

# Validate that non-missing values are whole numbers
for column in integer_columns:
    values = products_load_df[column].dropna()

    if not (values % 1 == 0).all():
        raise ValueError(
            f"Non-integer values found in {column}"
        )

# Convert columns to nullable integers
for column in integer_columns:
    products_load_df[column] = (
        products_load_df[column].astype("Int64")
    )

# Display the updated data types
print(products_load_df.dtypes)


In [ ]:

from sqlalchemy import text

# Check whether products already exist in PostgreSQL
with engine.connect() as connection:
    existing_count = connection.execute(
        text("SELECT COUNT(*) FROM olist.products")
    ).scalar_one()

print("Existing PostgreSQL products:", existing_count)

# Prevent accidental duplicate loads
if existing_count != 0:
    raise ValueError("Products table is not empty.")

# Validate source records
if len(products_load_df) != 32951:
    raise ValueError("Unexpected product count.")

if products_load_df["product_id"].isna().any():
    raise ValueError("Missing product IDs found.")

if products_load_df["product_id"].duplicated().any():
    raise ValueError("Duplicate product IDs found.")

# Load the records into the existing PostgreSQL table
with engine.begin() as connection:
    products_load_df.to_sql(
        name="products",
        con=connection,
        schema="olist",
        if_exists="append",
        index=False,
        chunksize=1000
    )

print("Products loaded successfully!")
